# S&P 500 協整法配對交易 (Cointegration Method)
## 基於 Johansen 協整檢定的嚴格實作

本 Notebook 回測協整法策略，完全遵守以下規則：

**一、 策略核心架構與時間視窗**
- 滾動窗口（Rolling Window）機制，更新頻率 21 個交易日（約一個月）。
- 形成期 (Formation Period)：252 個交易日（約一年）。
- 交易期 (Trading Period)：126 個交易日（約半年）。
- 隨時維持 6 個重疊組合（$126 \div 21 = 6$），各佔資金 $1/6$。
- 使用 S&P 500 成分股（含息報價）。

**二、 第一階段：配對篩選**
- 形成期首日價格正規化為 1。
- 進行 Johansen 協整檢定，依 Trace Statistics 最高分選前 20 對。
- 參數估計：$P_{1,t} - \beta P_{2,t} = \mu + \epsilon_t$，取得 $\beta$、$\mu$、$\sigma$。

**三、 第二階段：交易規則**
- 交易期首日將價格再次縮放為 1。
- 殘差 $Spread > \mu + 2\sigma$ $\rightarrow$ 賣空 1，買入 2。
- 殘差 $Spread < \mu - 2\sigma$ $\rightarrow$ 買入 1，賣空 2。
- 依 $\beta$ 比例建倉。
- 出場點：回歸均值 $\mu$ 收斂，或第 126 天強制平倉。

**四、 資金補填 (Market Fill)**
- 每個子組合等權重分配前 20 對。
- 若具協整關係少於 10 對，剩餘未使用資金直接做多 SPY (S&P 500)。

**五、 成本基準**
- 單邊交易與滑價：0.3% (30 bps)。
- 放空借券：年化 1%。


In [1]:
# 套件安裝與導入
import subprocess, sys
for pkg in ['statsmodels', 'plotly', 'kaleido', 'yfinance', 'joblib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, HTML
import yfinance as yf
import time
import math
import os

pd.set_option('display.float_format', '{:.4f}'.format)
print('✓ 套件導入完成')

✓ 套件導入完成


In [2]:
# ========== 階段 1：數據預處理與正規化 ==========
# === 全局參數設置 ===
FAST_TEST_MODE = True

if FAST_TEST_MODE:
    print("【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = 'Information Technology'  
else:
    print("【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# 資料庫配置
DB_PATH = r'..\data\SP500.db'
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'
USE_DYNAMIC_SECTORS = True  # 是否使用動態產業補齊

# 視窗參數 
FORMATION_WINDOW = 252    # 約 1 年
TRADING_WINDOW = 126      # 約 6 個月
ROLLING_WINDOW = 21       # 約 1 個月 (梯隊步長)
NUM_SUBPORTFOLIOS = math.ceil(TRADING_WINDOW / ROLLING_WINDOW)
MIN_HISTORY_DAYS = 200    # 最少歷史資料

# 配對篩選參數
TOP_N_PAIRS = 20          # 每個視窗選出前 N 組配對
SECTOR_NEUTRAL = True     # 行業中性化
MIN_VALID_PAIRS = 10    # 觸發 Market Fill 門檻

# 交易參數
Z_ENTRY = 2.0             # 進場標準差閾值（SSD 方法中改用 2×σ_formation）
Z_EXIT = 0.0              # 平倉標準差閾值
TRANSACTION_COST = 0.0030 # 雙邊交易成本 (0.29%)
MAX_LOSS_PCT = 0          # 停損 (0 = 無停損)


# 資金配置
INITIAL_CAPITAL = 10000  # 初始本金
SHORT_FEE_ANNUAL = 0.01   # 放空年化費率

# 確保路徑存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

print(f"""
【回測參數設置】
時間期間: {START_DATE} ~ {END_DATE}
形成期: {FORMATION_WINDOW} 天（約 1 年）
交易期: {TRADING_WINDOW} 天（約 6 個月）
步長: {ROLLING_WINDOW} 天（約 1 個月）
配對數: {TOP_N_PAIRS}
行業中性: {SECTOR_NEUTRAL}
初始資金: ${INITIAL_CAPITAL:,.0f}
交易成本: {TRANSACTION_COST*100:.2f}%
""")

【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。

【回測參數設置】
時間期間: 2019-01-01 ~ 2022-12-31
形成期: 252 天（約 1 年）
交易期: 126 天（約 6 個月）
步長: 21 天（約 1 個月）
配對數: 20
行業中性: True
初始資金: $10,000
交易成本: 0.30%



In [3]:
# === 數據加載函數 ===
def load_data_from_db(db_path, start_date, end_date):
    """從 SQLite 資料庫加載股票價格和行業分類"""
    conn = sqlite3.connect(db_path)
    
    # 嘗試多個可能的表名
    price_queries = [
        (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, close FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'✓ 加載價格數據：{len(prices_df):,} 筆記錄')
                break
        except Exception as e:
            print(f"❌ 讀取資料庫時發生錯誤: {e}")
            continue
    
    if prices_df is None or len(prices_df) == 0:
        raise RuntimeError("無法從資料庫加載價格數據")
    
    # 加載行業分類
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'✓ 加載行業數據：{len(sector_df):,} 檔股票')
                break
        except Exception:
            continue
    
    conn.close()
    
    if sector_df is None or len(sector_df) == 0:
        print('⚠️  未找到行業表，使用 Unknown 代替')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
    
    return prices_df, sector_df

def fix_unknown_sectors(sector_df, use_dynamic=USE_DYNAMIC_SECTORS, save_path=r'data\imputed_sectors.csv'):
    """具備本機快取與全域開關控制的產業補齊模組"""
    # 1. 開關判斷：如果不使用動態補齊，直接原封不動回傳
    if not use_dynamic:
        print("不使用動態產業補齊。")
        return sector_df

    # 確保儲存的目錄 (data\) 存在
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # 2. 快取讀取：如果已經抓過並存檔，直接載入
    if os.path.exists(save_path):
        print(f"從本機快取載入已補齊的產業分類: {save_path}")
        cached_df = pd.read_csv(save_path)
        update_df = cached_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        return sector_df.reset_index()

    # 3. API 抓取：如果沒有快取，執行連線作業
    unknown_mask = sector_df['sector'] == 'Unknown'
    unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
    
    if not unknown_tickers:
        return sector_df

    print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔股票的產業分類...")
    
    yf_logger = logging.getLogger('yfinance')
    original_level = yf_logger.level
    yf_logger.setLevel(logging.CRITICAL) 
    
    fixed_sectors = []
    
    for i, ticker in enumerate(unknown_tickers):
        try:
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            fixed_sectors.append({'ticker': ticker, 'sector': sector})
            time.sleep(0.02) 
        except Exception:
            fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
        if (i + 1) % 50 == 0:
            print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
            
    yf_logger.setLevel(original_level)
    
    # 4. 儲存快取：將剛抓下來的資料存成 CSV，下次就不用再抓了
    fetched_df = pd.DataFrame(fixed_sectors)
    fetched_df.to_csv(save_path, index=False)
    print(f"API 抓取完畢！已將動態產業分類永久儲存至: {save_path}")
    
    # 更新回原本的 DataFrame
    update_df = fetched_df.set_index('ticker')
    sector_df = sector_df.set_index('ticker')
    sector_df.update(update_df)
    sector_df = sector_df.reset_index()
    
    remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
    print(f"補齊完成！剩餘真實無法識別(已下市)的股票數量: {remaining}")
    
    return sector_df


def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """
    數據預處理：樞紐、前向填充、去除稀疏股票
    
    檢核清單項：
    ✓ 缺失值填充（最多 5 天前向填充）
    ✓ 去除歷史資料不足的股票
    ✓ 對齊時間序列
    """
    # 樞紐表：時間 × 股票
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    
    # 前向填充（最多 5 天）
    pivot.ffill(limit=5, inplace=True)
    
    # 保留有足夠歷史資料的股票
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    
    print(f'✓ 數據矩陣：{len(pivot)} 天 × {len(pivot.columns)} 檔股票')
    print(f'✓ 時間範圍：{pivot.index[0].date()} ~ {pivot.index[-1].date()}')
    
    return pivot


# === 加載和預處理數據 ===
print("\n從資料庫加載原始數據...")
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)

print("\n預處理價格數據...")
price_pivot = preprocess_prices(prices_raw, min_days=MIN_HISTORY_DAYS)

# 建立行業對應字典
sector_map = sector_info.set_index('ticker')['sector'].to_dict()
print(f'\n【行業分佈】')
print(pd.Series(sector_map).value_counts().head(10))


從資料庫加載原始數據...
✓ 加載價格數據：620,483 筆記錄
⚠️  未找到行業表，使用 Unknown 代替

預處理價格數據...


KeyError: 'date'

In [ ]:
# 板塊過濾（如果設定了 TARGET_SECTOR）
if TARGET_SECTOR is not None:
    # 篩選出符合目標產業的股票 ticker
    target_tickers = sector_info[sector_info['sector'] == TARGET_SECTOR]['ticker'].tolist()
    print(f"🎯 板塊過濾：篩選 {TARGET_SECTOR} 板塊之股票，共 {len(target_tickers)} 檔")
    
    # 過濾 prices_raw：只保留目標產業的股票
    prices_raw = prices_raw[prices_raw['ticker'].isin(target_tickers)]
    
    # 過濾 sector_info：只保留目標產業的資訊
    sector_info = sector_info[sector_info['sector'] == TARGET_SECTOR]
else:
    print("🌍 全市場模式：使用全部 S&P 500 股票")

# ============================================================

# 
price_pivot= preprocess_prices(prices_raw)
sector_map = sector_info.set_index('ticker')['sector'].to_dict()

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.vector_ar.vecm import coint_johansen
# === 選擇與計算 Johansen 協整 ===
def perform_johansen_selection(form_prices, top_n=TOP_N_PAIRS):
    """
    一、 正規化：形成期首日價格設為 1
    二、 Johansen Trace Stat > 95% 且數值最高前 20 對
    三、 估計 P1 - beta*P2 = mu + e
    """
    form_prices = form_prices.dropna(axis=1)
    if form_prices.shape[1] < 2: return []
    
    # 1. 價格歸一化 (首日=1)
    norm_p = (form_prices / form_prices.iloc[0]).dropna(axis=1)
    cols = norm_p.columns.tolist()
    
    results = []
    
    # 快速過濾：同向變動相關性大於 0.5 才做 Johansen，降低運算量
    corr_matrix = norm_p.corr()
    pairs = []
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            if corr_matrix.iloc[i, j] > 0.5:  
                pairs.append((cols[i], cols[j]))
                
    for c1, c2 in pairs:
        y = norm_p[[c1, c2]].values
        
        # Johansen Cointegration
        try:
            res = coint_johansen(y, det_order=0, k_ar_diff=1)
            trace_stat = res.lr1[0]
            trace_crit = res.cvt[0, 1]  # 95% 臨界值
            
            if trace_stat > trace_crit: 
                # 協整成立
                P1 = norm_p[c1]
                P2 = norm_p[c2]
                P2_const = sm.add_constant(P2)
                ols_res = sm.OLS(P1, P2_const).fit()
                beta = ols_res.params[c2]
                
                # 計算殘差 (Spread = P1 - beta*P2)
                spread = P1 - beta * P2
                mu = spread.mean()
                sigma = spread.std()
                
                results.append({
                    'pair': (c1, c2),
                    'trace_stat': trace_stat,
                    'beta': beta,
                    'mu': mu,
                    'sigma': sigma
                })
        except:
            pass
            
    df_res = pd.DataFrame(results)
    if df_res.empty: return []
    
    # Trace Stat 由大到小排序
    df_res = df_res.sort_values('trace_stat', ascending=False).head(top_n)
    return df_res.to_dict('records')


In [ ]:
# === 交易邏輯與資本狀態機 ===
def run_trading_period(trade_prices, bench_series, config_pairs, initial_sub_capital=INITIAL_CAPITAL/NUM_SUBPORTFOLIOS):
    """
    一、 正規化：交易期開始首日再次縮放為 1
    二、 進出場門檻：2-SD 觸發
    三、 若配對數不足，剩餘分配給 Market Fill
    """
    dates = trade_prices.index
    valid_pairs_count = len(config_pairs)
    slots = max(TOP_N_PAIRS, valid_pairs_count)
    capital_per_slot = initial_sub_capital / slots
    
    # Market Fill (缺失部位投入大盤)
    market_fill_capital = 0.0
    if valid_pairs_count < MIN_VALID_PAIRS:
        market_fill_capital = (slots - valid_pairs_count) * capital_per_slot
        
    portfolio_value = pd.Series(0.0, index=dates)
    
    # 1. Market Fill 預算放入大盤
    if market_fill_capital > 0:
        b0 = bench_series.iloc[0]
        spy_shares = (market_fill_capital * (1 - TRANSACTION_COST)) / b0
        portfolio_value += bench_series * spy_shares
    elif valid_pairs_count == 0:
        return pd.Series(initial_sub_capital, index=dates)

    # 2. 配對交易部分
    for cfg in config_pairs:
        c1, c2 = cfg['pair']
        if c1 not in trade_prices.columns or c2 not in trade_prices.columns:
            portfolio_value += capital_per_slot
            continue
            
        p1 = trade_prices[c1]
        p2 = trade_prices[c2]
        
        # 交易期二次歸一化
        p1_0 = p1.iloc[0]
        p2_0 = p2.iloc[0]
        if p1_0 == 0 or p2_0 == 0 or pd.isna(p1_0) or pd.isna(p2_0):
            portfolio_value += capital_per_slot
            continue
            
        norm_p1 = p1 / p1_0
        norm_p2 = p2 / p2_0
        
        beta, mu, sigma = cfg['beta'], cfg['mu'], cfg['sigma']
        spread = norm_p1 - beta * norm_p2
        
        position = 0 # 0:空 , 1:做多1做空2, -1:做空1做多2
        shares1_norm = 0
        shares2_norm = 0
        current_pair_cash = capital_per_slot
        pair_value_daily = np.zeros(len(dates))
        
        threshold_up = mu + Z_ENTRY * sigma
        threshold_dn = mu - Z_ENTRY * sigma
        
        for t in range(len(dates)):
            val1, val2 = p1.iloc[t], p2.iloc[t]
            n_val1, n_val2 = norm_p1.iloc[t], norm_p2.iloc[t]
            sp = spread.iloc[t]
            
            # 放空成本
            if position != 0:
                short_fee = (current_pair_cash / 2) * (SHORT_FEE_ANNUAL / 252)
                current_pair_cash -= short_fee
                
            if position == 0:
                if sp > threshold_up:
                    position = -1 # 賣出(放空) 1，買入 2
                elif sp < threshold_dn:
                    position = 1  # 買入 1，賣出(放空) 2
                    
                if position != 0: # 建倉
                    # 依 Beta 設定等價值對等部位，這裡以簡化50/50保證金資金計算
                    side_cap = current_pair_cash / 2
                    
                    if position == -1: # 空1多2
                        shares1_norm = -(side_cap * (1 - TRANSACTION_COST)) / n_val1
                        shares2_norm = (side_cap * (1 - TRANSACTION_COST)) / n_val2
                    else: # 多1空2
                        shares1_norm = (side_cap * (1 - TRANSACTION_COST)) / n_val1
                        shares2_norm = -(side_cap * (1 - TRANSACTION_COST)) / n_val2
                        
                    current_pair_cash -= current_pair_cash * TRANSACTION_COST 
            else:
                # 檢查平倉
                exit_signal = False
                if position == -1 and sp <= mu: exit_signal = True
                elif position == 1 and sp >= mu: exit_signal = True
                elif t == len(dates) - 1: exit_signal = True # 強制平倉
                
                if exit_signal:
                    val_1_now = shares1_norm * n_val1
                    val_2_now = shares2_norm * n_val2
                    position_gross = abs(val_1_now) + abs(val_2_now)
                    cost = position_gross * TRANSACTION_COST
                    
                    current_pair_cash += (val_1_now + val_2_now - cost)
                    
                    position, shares1_norm, shares2_norm = 0, 0, 0
                    
            if position == 0:
                 pair_value_daily[t] = current_pair_cash
            else:
                 pair_value_daily[t] = current_pair_cash + (shares1_norm * n_val1) + (shares2_norm * n_val2)
                
        portfolio_value += pair_value_daily
        
    return portfolio_value


## 階段 4：滾動視窗回測框架

### 檢核清單：
- ✓ **形成期**：252 天（約 1 年）
- ✓ **交易期**：126 天（約 6 個月）
- ✓ **滾動步長**：20 天（約 1 個月）
- ✓ **梯隊資金**：將資本分割到每個滾動視窗
- ✓ **聚合損益**：逐日累積所有視窗的損益

In [ ]:
# ========== 階段 4：滾動視窗回測框架 ==========
def run_coin_backtest(price_pivot, sector_map,
                     formation_window=FORMATION_WINDOW,
                     trading_window=TRADING_WINDOW,
                     rolling_window=ROLLING_WINDOW,
                     top_n=TOP_N_PAIRS,
                     initial_capital=INITIAL_CAPITAL):
    """
    完整的滾動視窗 SSD 配對交易回測
    
    檢核清單項：
    ✓ 形成期 = 252 天
    ✓ 交易期 = 126 天
    ✓ 滾動步長 = 20 天
    ✓ 梯隊資金管理
    ✓ 日度損益聚合
    """
    
    dates = price_pivot.index
    N = len(dates)
    
    # ===== 初始化 =====
    all_pnl = pd.DataFrame(index=dates, dtype=float)
    all_trades = []
    window_records = []
    
    num_tranches = math.ceil(trading_window / rolling_window) + 1
    tranche_capital = initial_capital / num_tranches
    
    print(f"\n【回測配置】")
    print(f"  資本梯隊數: {num_tranches}")
    print(f"  每梯隊配置: ${tranche_capital:,.2f}")
    print(f"  時間跨度: {dates[0].date()} ~ {dates[-1].date()}")
    
    # ===== 滾動視窗迴圈 =====
    si, window_id = 0, 0
    
    while si + formation_window + trading_window <= N:
        fe = si + formation_window        # 形成期結束
        te = fe + trading_window          # 交易期結束
        
        form_prices = price_pivot.iloc[si:fe]
        trade_prices = price_pivot.iloc[fe:te]
        
        # 配對篩選
        pairs = perform_johansen_selection(form_prices, sector_map, top_n=top_n)
        
        if not pairs:
            si += rolling_window
            window_id += 1
            continue
        
        # 本視窗的損益
        window_pnl = pd.Series(0.0, index=dates)
        window_all_trades = []
        
        # 對每個配對執行交易
        for pair in pairs:
            pair_cap = tranche_capital / len(pairs) if len(pairs)>0 else 0
            result = run_trading_period(trade_prices, pair, form_prices, pair_capital=pair_cap)
            
            pair_pnl = result['pnl']  # 已經是美元絕對值
            window_pnl = window_pnl.add(pair_pnl, fill_value=0.0)
            window_all_trades.extend(result['trades'])
        
        # 記錄本視窗
        col_name = f'W{window_id:03d}'
        all_pnl[col_name] = window_pnl
        all_trades.extend(window_all_trades)
        
        # 計算本視窗績效
        active_pnl = window_pnl[(window_pnl.index >= dates[fe]) & 
                                (window_pnl.index <= dates[te-1])]
        window_return = active_pnl.sum() / tranche_capital if tranche_capital > 0 else 0.0
        trade_count = len([t for t in window_all_trades 
                          if pd.to_datetime(t['entry_date']) >= dates[fe]])
        
        window_records.append({
            'window_id': window_id,
            'form_start': dates[si],
            'form_end': dates[fe-1],
            'trade_start': dates[fe],
            'trade_end': dates[te-1],
            'pairs_count': len(pairs),
            'trade_count': trade_count,
            'return_pct': window_return * 100,
            'pnl_total': active_pnl.sum()
        })
        
        # 列印進度
        print(f"  W{window_id:03d} | {dates[fe].date()} ~ {dates[te-1].date()} | "
                  f"配對: {len(pairs)} | 交易: {trade_count} | 報酬: {window_return*100:+.2f}%")
        
        si += rolling_window
        window_id += 1
    
    print(f"\n✓ 回測完成：{window_id} 個視窗，{len(all_trades)} 筆交易")
    
    return {
        'pnl': all_pnl,
        'trades': all_trades,
        'windows': window_records
    }

# ===== 執行回測 =====
print("\n執行滾動視窗回測...")
backtest_result = run_coin_backtest(price_pivot, sector_map)

## 階段 5：績效指標計算與分析

### 計算指標：
- **CAGR (年化報酬率)**：$\text{CAGR} = (1 + R)^{252/n} - 1$
- **年化波動率**：$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$
- **Sharpe Ratio**：$SR = \frac{r_{annual}}{\sigma_{annual}}$
- **Sortino Ratio**：$\text{Sortino} = \frac{r_{annual}}{\sigma_{downside}}$
- **最大回撤 (MDD)**：$\text{MDD} = \min\left(\frac{V_t - V_{max}}{V_{max}}\right)$
- **勝率**：$\frac{\text{正報酬日數}}{\text{總交易日數}}$
- **平均獲利/虧損**：配對層級統計

In [ ]:
# ========== 階段 5：績效指標計算與分析 ==========
def compute_performance_metrics(pnl_df, initial_capital, all_trades):
    """計算完整的績效指標"""
    
    # ===== 計算組合損益和報酬率 =====
    portfolio_pnl = pnl_df.sum(axis=1)
    returns = portfolio_pnl / initial_capital
    returns = returns.dropna().replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(returns) == 0:
        print("⚠️  無有效報酬數據")
        return None
    
    # ===== 淨值曲線 =====
    nav = (1 + returns).cumprod()
    
    # ===== 基本指標 =====
    total_return = nav.iloc[-1] - 1
    cagr = (1 + total_return) ** (252 / len(returns)) - 1 if total_return > -1 else -1
    annual_vol = returns.std() * np.sqrt(252)
    
    # ===== 比率指標 =====
    sharpe = (returns.mean() * 252) / annual_vol if annual_vol > 1e-8 else np.nan
    
    # Sortino Ratio（僅計算下行波動率）
    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    sortino = (returns.mean() * 252) / downside_vol if downside_vol > 1e-8 else np.nan
    
    # ===== 風險指標 =====
    drawdown = (nav - nav.cummax()) / nav.cummax()
    mdd = drawdown.min()
    
    # 恢復時間
    max_dd_date = drawdown.idxmin()
    recovery_dates = nav[nav.index > max_dd_date][nav >= nav[nav.index <= max_dd_date].max()]
    recovery_days = (recovery_dates.index[0] - max_dd_date).days if len(recovery_dates) > 0 else np.nan
    
    # ===== 交易統計 =====
    win_rate = (returns > 0).sum() / len(returns)
    
    # 配對層級統計
    if all_trades:
        trades_df = pd.DataFrame(all_trades)
        avg_profit = trades_df['profit'].mean()
        avg_hold = trades_df['hold_days'].mean()
        win_trades = (trades_df['profit'] > 0).sum()
        total_trades = len(trades_df)
        trade_win_rate = win_trades / total_trades if total_trades > 0 else 0
        avg_profit_win = trades_df[trades_df['profit'] > 0]['profit'].mean() if win_trades > 0 else 0
        avg_profit_loss = trades_df[trades_df['profit'] <= 0]['profit'].mean() if total_trades - win_trades > 0 else 0
    else:
        avg_profit = avg_hold = win_trades = total_trades = trade_win_rate = 0
        avg_profit_win = avg_profit_loss = 0.0
    
    metrics = {
        'CAGR(%)': round(cagr * 100, 2),
        '總報酬(%)': round(total_return * 100, 2),
        '年化波動(%)': round(annual_vol * 100, 2),
        'Sharpe': round(sharpe, 4),
        'Sortino': round(sortino, 4),
        '最大回撤(%)': round(mdd * 100, 2),
        '恢復天數': round(recovery_days, 0) if not np.isnan(recovery_days) else float('nan'),
        '日勝率(%)': round(win_rate * 100, 2),
        '配對勝率(%)': round(trade_win_rate * 100, 2),
        '平均配對利潤': round(avg_profit, 4),
        '平均獲利配對': round(avg_profit_win, 4),
        '平均虧損配對': round(avg_profit_loss, 4),
        '平均持倉天數': round(avg_hold, 1),
        '總配對數': total_trades
    }
    
    return metrics

# ===== 計算績效 =====
print("\n計算績效指標...")
pnl_df = backtest_result['pnl']
trades = backtest_result['trades']

metrics = compute_performance_metrics(pnl_df, INITIAL_CAPITAL, trades)

if metrics:
    print("\n【核心績效指標】")
    print(f"  CAGR: {metrics['CAGR(%)']:.2f}%")
    print(f"  年化波動: {metrics['年化波動(%)']:.2f}%")
    print(f"  Sharpe: {metrics['Sharpe']:.4f}")
    print(f"  最大回撤: {metrics['最大回撤(%)']:.2f}%")
    print(f"  日勝率: {metrics['日勝率(%)']:.2f}%")
    print(f"  配對勝率: {metrics['配對勝率(%)']:.2f}%")
    print(f"  總配對交易: {metrics['總配對數']}")
    
    # 詳細指標表
    metrics_df = pd.DataFrame([metrics])
    display(metrics_df.style.format("{:.2f}", na_rep="N/A"))

## 階段 6：結果視覺化與對標

### 視覺化內容：
1. **淨值曲線 (NAV)** - 累積報酬走勢
2. **回撤曲線** - 最大回撤動態
3. **日度報酬分佈** - 報酬率直方圖
4. **月度績效熱力圖** - 時間分解
5. **配對價差時序** - 交易信號驗證
6. **交易日誌表** - 詳細交易記錄
7. **對標比較** - vs. S&P 500

In [ ]:
import yfinance as yf
import pandas as pd

def fetch_benchmark_returns(target_index, ticker="SPY"):
    """
    Fetch benchmark daily returns and align with the strategy's trading days.
    Default ticker is Taiwan Taiex (^TWII). Use 'SPY' or '^GSPC' for US market.
    """
    # Extract start and end dates from the strategy's index
    start_date = target_index.min().strftime('%Y-%m-%d')
    # Add one day to end_date to ensure the last day is included in yfinance
    end_date = (target_index.max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    
    print(f"Downloading benchmark data ({ticker}) from {start_date} to {end_date}...")
    
    # Download daily data
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    if df.empty:
        print("Warning: Failed to fetch benchmark data. Returning zeros.")
        return pd.Series(0, index=target_index)
    
    # Calculate daily returns based on Adjusted Close
    # Handle multi-index columns if yfinance version >= 0.2.40 returns them
    if isinstance(df.columns, pd.MultiIndex):
        if 'Adj_Close' not in df.columns:
            if 'Close' in df.columns:
                close_prices = df['Close'][ticker]
    else:
        if 'Adj_Close' not in df.columns:
            if 'Close' in df.columns:
                close_prices = df['Close']
        
    bench_returns = close_prices.pct_change().dropna()
    
    # Align benchmark returns with the strategy's datetime index
    # Fill missing dates (e.g., market holidays) with 0 return to avoid NaN issues
    aligned_returns = bench_returns.reindex(target_index).fillna(0)
    
    return aligned_returns

# ==========================================
# Integration with Step 6.1
# ==========================================

# 1. Fetch the benchmark returns using your strategy's date index
# Note: 'returns' is the variable you calculated in Step 5 (portfolio_pnl / INITIAL_CAPITAL)
benchmark_returns = fetch_benchmark_returns(returns.index, ticker="^TWII")

# 2. Calculate the Cumulative Net Asset Value (NAV) for the benchmark
benchmark_nav = (1 + benchmark_returns).cumprod()

print("✓ Benchmark data successfully loaded and aligned.")

In [ ]:
# ========== 階段 6：結果視覺化與對標 ==========
# ===== 準備數據 =====
portfolio_pnl = pnl_df.sum(axis=1)
returns = portfolio_pnl / INITIAL_CAPITAL
returns = returns.dropna().replace([np.inf, -np.inf], np.nan).dropna()
nav = (1 + returns).cumprod()
drawdown = (nav - nav.cummax()) / nav.cummax() * 100

# ===== 1. 淨值曲線與回撤 =====
print("\n【第 6.1 步】繪製淨值曲線與回撤 (包含大盤對標)...")

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['淨值曲線 (NAV) 比較', '最大回撤 (MDD)'],
    vertical_spacing=0.1
)

# Plot Strategy NAV
fig.add_trace(
    go.Scatter(
        x=nav.index, y=nav.values,
        name='SSD Strategy',
        line=dict(color='#1f77b4', width=2),
        hovertemplate='%{x|%Y-%m-%d}<br>Strategy NAV: %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# Plot Benchmark NAV 
fig.add_trace(
    go.Scatter(
        x=benchmark_nav.index, y=benchmark_nav.values,
        name='Market Benchmark',
        line=dict(color='#ff7f0e', width=2, dash='dot'),
        hovertemplate='%{x|%Y-%m-%d}<br>Benchmark NAV: %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# 回撤
fig.add_trace(
    go.Scatter(
        x=drawdown.index, y=drawdown.values,
        name='MDD',
        fill='tozeroy',
        line=dict(color='#d62728', width=1),
        hovertemplate='%{x|%Y-%m-%d}<br>回撤: %{y:.2f}%<extra></extra>'
    ),
    row=2, col=1
)

fig.update_layout(
    title='SSD 配對交易系統 - 淨值與回撤曲線 (2000-2025)',
    height=700,
    template='plotly_dark',
    hovermode='x unified',
    showlegend=True
)

fig.update_yaxes(title_text='NAV', row=1, col=1)
fig.update_yaxes(title_text='回撤 (%)', row=2, col=1)
fig.show()

print("✓ 淨值曲線已繪製")

# ===== 2. 日度報酬分佈 =====
print("\n【第 6.2 步】繪製報酬分佈...")

fig_returns = go.Figure()

fig_returns.add_trace(go.Histogram(
    x=returns * 100,
    name='日度報酬',
    nbinsx=50,
    marker=dict(color='#2ca02c', opacity=0.7),
    hovertemplate='報酬率: %{x:.2f}%<br>頻次: %{y}<extra></extra>'
))

fig_returns.add_vline(
    x=returns.mean() * 100,
    line_dash='dash',
    line_color='red',
    annotation_text=f"平均: {returns.mean()*100:.3f}%",
    annotation_position="top right"
)

fig_returns.update_layout(
    title='日度報酬率分佈',
    xaxis_title='報酬率 (%)',
    yaxis_title='頻次',
    height=400,
    template='plotly_dark'
)

fig_returns.show()

print("✓ 報酬分佈已繪製")

# ===== 3. 月度績效熱力圖 =====
print("\n【第 6.3 步】生成月度績效熱力圖...")

# 按月計算報酬
returns_monthly = returns.resample('M').apply(lambda x: (1 + x).prod() - 1)
returns_monthly.index = returns_monthly.index.to_period('M')

# 轉換為年-月矩陣
if len(returns_monthly) > 0:
    pivot_returns = pd.DataFrame({
        'year': [d.year for d in returns_monthly.index],
        'month': [d.month for d in returns_monthly.index],
        'return': returns_monthly.values
    })
    
    pivot_table = pivot_returns.pivot(index='year', columns='month', values='return') * 100
    
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=pivot_table.values,
        x=['1月', '2月', '3月', '4月', '5月', '6月', 
           '7月', '8月', '9月', '10月', '11月', '12月'],
        y=pivot_table.index,
        colorscale='RdYlGn',
        zmid=0,
        hovertemplate='%{y}年 %{x}: %{z:.2f}%<extra></extra>'
    ))
    
    fig_heatmap.update_layout(
        title='月度報酬率熱力圖 (%)',
        xaxis_title='月份',
        yaxis_title='年份',
        height=500,
        template='plotly_dark'
    )
    
    fig_heatmap.show()
    print("✓ 熱力圖已繪製")

# ===== 4. 視窗績效統計 =====
print("\n【第 6.4 步】視窗績效統計...")

windows_df = pd.DataFrame(backtest_result['windows'])
if len(windows_df) > 0:
    print(f"\n【視窗績效摘要】(共 {len(windows_df)} 個視窗)")
    print(windows_df[['window_id', 'pairs_count', 'trade_count', 'return_pct']].head(10)
                    .to_string(index=False))
    
    print(f"\n  平均配對數: {windows_df['pairs_count'].mean():.1f}")
    print(f"  平均交易數: {windows_df['trade_count'].mean():.1f}")
    print(f"  平均視窗報酬: {windows_df['return_pct'].mean():+.2f}%")
    print(f"  正報酬視窗: {(windows_df['return_pct'] > 0).sum()}/{len(windows_df)}")

# ===== 5. 交易日誌 =====
print("\n【第 6.5 步】交易日誌統計...")

if trades:
    trades_df = pd.DataFrame(trades)
    trades_df['entry_date'] = pd.to_datetime(trades_df['entry_date'])
    trades_df['exit_date'] = pd.to_datetime(trades_df['exit_date'])
    
    print(f"\n【交易統計】(共 {len(trades_df)} 筆交易)")
    print(f"  獲利交易: {(trades_df['profit'] > 0).sum()}")
    print(f"  虧損交易: {(trades_df['profit'] <= 0).sum()}")
    print(f"  平均利潤: {trades_df['profit'].mean():.6f}")
    print(f"  最大利潤: {trades_df['profit'].max():.6f}")
    print(f"  最大虧損: {trades_df['profit'].min():.6f}")
    print(f"  平均持倉: {trades_df['hold_days'].mean():.1f} 天")
    
    # 按平倉原因分類
    print(f"\n【平倉原因統計】")
    exit_reasons = trades_df['exit_reason'].value_counts()
    for reason, count in exit_reasons.items():
        pct = count / len(trades_df) * 100
        print(f"  {reason}: {count} ({pct:.1f}%)")
    
    # 前 10 筆交易
    print(f"\n【前 10 筆交易記錄】")
    display(trades_df[['stock_a', 'stock_b', 'entry_date', 'exit_date', 
                       'entry_spread', 'exit_spread', 'profit', 'hold_days']].head(10)
                    .style.format({'profit': '{:.6f}', 'entry_spread': '{:.4f}', 
                                 'exit_spread': '{:.4f}'}))

print("\n✓ 視覺化與分析完成")

### 【年淨值走勢圖】
按年份展示淨值變化，清晰呈現策略在不同年度的表現。

In [ ]:
# ===== 年淨值走勢圖 =====
print("\n【第 6.2.5 步】繪製年淨值走勢圖...")

# 計算每年度的淨值
nav_annual = nav.resample('Y').last()
nav_annual.index = nav_annual.index.year

# 初始值設置
nav_annual_values = [1.0]  # 2000年初始淨值
for i in range(len(nav_annual) - 1):
    nav_annual_values.append(nav_annual.iloc[i])

nav_annual_values = nav_annual.values
years = nav_annual.index.astype(str).astype(int).tolist()

# 繪製年淨值走勢折線圖
fig_nav_annual = go.Figure()

# 添加淨值線
fig_nav_annual.add_trace(go.Scatter(
    x=years,
    y=nav_annual_values,
    mode='lines+markers',
    name='年末淨值',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=8, symbol='circle'),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.2)',
    hovertemplate='%{x}年<br>淨值: %{y:.4f}<extra></extra>'
))

# 添加年初淨值 (初始值為1)
years_with_initial = [years[0] - 1] + years
nav_values_with_initial = [1.0] + nav_annual_values.tolist()

fig_nav_annual.add_trace(go.Scatter(
    x=years_with_initial,
    y=nav_values_with_initial,
    mode='lines+markers',
    name='累積淨值',
    line=dict(color='#ff7f0e', width=2, dash='dash'),
    marker=dict(size=6),
    hovertemplate='%{x}年<br>累積淨值: %{y:.4f}<extra></extra>',
    showlegend=False
))

# 更新佈局
fig_nav_annual.update_layout(
    title='SSD 配對交易系統 - 年淨值走勢圖 (2000-2025)',
    xaxis_title='年份',
    yaxis_title='淨值',
    height=500,
    template='plotly_dark',
    hovermode='x unified',
    xaxis=dict(
        tickmode='linear',
        tick0=2000,
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128, 128, 128, 0.2)'
    ),
    font=dict(size=11),
    showlegend=True,
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(0, 0, 0, 0.5)'
    )
)

# 添加數值標籤
for year, nav_val in zip(years, nav_annual_values):
    fig_nav_annual.add_annotation(
        x=year,
        y=nav_val,
        text=f'{nav_val:.2f}',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor='#1f77b4',
        ax=0,
        ay=-20,
        font=dict(size=9, color='#1f77b4')
    )

fig_nav_annual.show()

print("✓ 年淨值走勢圖已繪製")

# 計算年度變化統計
print("\n【年度淨值統計】")
print(f"{'年份':<8} {'年末淨值':<12} {'年度報酬(%)':<12} {'累計報酬(%)':<12}")
print("-" * 46)

prev_nav = 1.0
for year, nav_val in zip(years, nav_annual_values):
    annual_return = (nav_val - prev_nav) / prev_nav * 100
    cumulative_return = (nav_val - 1) * 100
    print(f"{year:<8} {nav_val:<12.4f} {annual_return:>10.2f}% {cumulative_return:>10.2f}%")
    prev_nav = nav_val

# 最佳年份和最差年份
best_year_idx = np.argmax(nav_annual_values)
worst_year_idx = np.argmin(nav_annual_values)

print(f"\n【極值統計】")
print(f"  最佳年份: {years[best_year_idx]} (淨值: {nav_annual_values[best_year_idx]:.4f})")
print(f"  最差年份: {years[worst_year_idx]} (淨值: {nav_annual_values[worst_year_idx]:.4f})")
print(f"  年均淨值: {np.mean(nav_annual_values):.4f}")
print(f"  年末淨值: {nav_annual_values[-1]:.4f} (總增長: {(nav_annual_values[-1]-1)*100:.2f}%)")

## 進階分析：時間與部門分解

## 數據匯出與存檔

In [ ]:
# === 匯出回測結果 ===
print("\n【數據匯出】")

# 確保 results 目錄存在
os.makedirs('..\results', exist_ok=True)

# 1. 導出 PnL
pnl_export = pnl_df.sum(axis=1)
pnl_export.to_csv('results/ssd_pnl.csv')
print("✓ 已匯出日度損益 -> results/ssd_pnl.csv")

# 2. 導出淨值
nav_export = (1 + (pnl_export / INITIAL_CAPITAL)).cumprod()
nav_export.to_csv('results/ssd_nav.csv')
print("✓ 已匯出淨值曲線 -> results/ssd_nav.csv")

# 3. 導出交易日誌
if trades:
    trades_df_export = pd.DataFrame(trades)
    trades_df_export.to_csv('results/ssd_trades.csv', index=False)
    print(f"✓ 已匯出交易日誌 ({len(trades_df_export)} 筆) -> results/ssd_trades.csv")

# 4. 導出視窗績效
windows_df_export = pd.DataFrame(backtest_result['windows'])
windows_df_export.to_csv('results/ssd_windows.csv', index=False)
print(f"✓ 已匯出視窗績效 ({len(windows_df_export)} 個) -> results/ssd_windows.csv")

# 5. 導出績效指標
if metrics:
    pd.DataFrame([metrics]).to_csv('results/ssd_metrics.csv', index=False)
    print("✓ 已匯出績效指標 -> results/ssd_metrics.csv")

print("\n✅ 回測完整！所有結果已保存到 results/ 目錄")